In [1]:
import pandas as pd

In [2]:
# Load the data
df = pd.read_csv('data/netflix_shows_complete.csv')

In [3]:
# Check missing data percentage
missing_pct = (df.isnull().sum()/len(df))* 100
print(missing_pct[missing_pct>0].sort_values(ascending=False))

avg_episode_runtime        49.600000
created_by                 31.133333
us_content_rating          14.266667
keywords                    8.533333
homepage                    1.733333
imdb_id                     1.533333
last_air_date               1.000000
days_since_last_episode     1.000000
first_air_date              0.533333
show_age_days               0.533333
genres                      0.333333
origin_country              0.066667
dtype: float64


In [4]:
# Fill categorical data with "Unknown"
df['created_by'] = df['created_by'].fillna('Unknown')
df['us_content_rating'] = df['us_content_rating'].fillna('Unknown')
df['keywords'] = df['keywords'].fillna('Unknown')
df['imdb_id'] = df['imdb_id'].fillna('Unknown')
df['homepage'] = df['homepage'].fillna('Unknown')
df['last_air_date'] = df['last_air_date'].fillna('Unknown')
df['first_air_date'] = df['first_air_date'].fillna('Unknown')

In [5]:
# Drop column with over 50% of missing data
df_new = df.drop('avg_episode_runtime', axis=1)

In [6]:
# Drop rows where minimal data is missing
df_new = df_new.dropna(subset = ['show_age_days', 'days_since_last_episode'])

In [7]:
print(f"After cleaning: {len(df_new)} shows remain")

After cleaning: 1485 shows remain


In [8]:
# Defining target variable for prediction
df_new['is_canceled'] = (df_new['status'] == 'Canceled').astype(int)

In [9]:
# Define weighted rating system using Bayesian Average
# C = mean vote_average across dataset, m = minimum votes threshold
C = df_new['vote_average'].mean()
m = 25
df_new['weighted_rating'] = (df_new['vote_count'] / (df_new['vote_count'] + m)) * df_new['vote_average'] + \
                             (m / (df_new['vote_count'] + m)) * C

In [10]:
print(df_new['weighted_rating'])

0       8.579231
1       8.434869
2       6.822324
3       7.146342
4       6.388981
          ...   
1493    7.282930
1494    7.161978
1497    6.988935
1498    6.685244
1499    6.949304
Name: weighted_rating, Length: 1485, dtype: float64


In [11]:
print(df_new['weighted_rating'].min())

3.2258319487712015


In [ ]:
# Categorize ratings (weighted_rating is on a 0-10 scale)
def rating_category(rating):
    if rating >= 8.0:
        return 'Excellent'
    elif rating >= 7.0:
        return 'Good'
    elif rating >= 6.0:
        return 'Average'
    else:
        return 'Poor'

In [13]:
df_new['rating_category'] = df_new['weighted_rating'].apply(rating_category)

In [14]:
print(df_new['rating_category'])

0       Excellent
1       Excellent
2         Average
3            Good
4         Average
          ...    
1493         Good
1494         Good
1497      Average
1498      Average
1499      Average
Name: rating_category, Length: 1485, dtype: object


In [15]:
# Show longevity
df_new['years_since_premiere'] = (df_new['show_age_days']/365)

In [16]:
print(df_new['years_since_premiere'])

0        9.539726
1       10.010959
2       33.063014
3        0.049315
4       28.032877
          ...    
1493     8.408219
1494     9.063014
1497     4.731507
1498     6.356164
1499     1.652055
Name: years_since_premiere, Length: 1485, dtype: float64


In [17]:
# Recent vs older shows
df_new['is_recent'] = (df_new['years_since_premiere'] < 5).astype(int)

In [18]:
print(df_new['is_recent'])

0       0
1       0
2       0
3       1
4       0
       ..
1493    0
1494    0
1497    1
1498    0
1499    1
Name: is_recent, Length: 1485, dtype: int64


In [19]:
# Has high popularity
df_new['is_popular'] = (df_new['popularity'] > df_new['popularity'].median()).astype(int)

In [20]:
print(df_new['is_popular'])

0       1
1       1
2       1
3       1
4       1
       ..
1493    0
1494    0
1497    0
1498    0
1499    0
Name: is_popular, Length: 1485, dtype: int64


In [21]:
# Number of genres
df_new['genre_count'] = df_new['genres'].apply(lambda x: len(str(x).split(',')) if pd.notna(x) else 0)

In [22]:
print(df_new['genre_count'])

0       3
1       2
2       1
3       3
4       4
       ..
1493    4
1494    2
1497    2
1498    2
1499    1
Name: genre_count, Length: 1485, dtype: int64


In [23]:
print(df_new['first_air_date'])

0       2016-07-15
1       2016-01-25
2       1993-01-11
3       2026-01-08
4       1998-01-21
           ...    
1493    2017-09-01
1494    2017-01-05
1497    2021-05-05
1498    2019-09-20
1499    2024-06-02
Name: first_air_date, Length: 1485, dtype: object


In [24]:
# Convert dates to datetime objects
df_new['first_air_date'] = pd.to_datetime(df_new['first_air_date'], format='%Y-%m-%d')
df_new['last_air_date'] = pd.to_datetime(df_new['last_air_date'], format='%Y-%m-%d')

In [25]:
print(df_new['first_air_date'])

0      2016-07-15
1      2016-01-25
2      1993-01-11
3      2026-01-08
4      1998-01-21
          ...    
1493   2017-09-01
1494   2017-01-05
1497   2021-05-05
1498   2019-09-20
1499   2024-06-02
Name: first_air_date, Length: 1485, dtype: datetime64[ns]


In [26]:
print(df_new['last_air_date'])

0      2025-12-31
1      2021-09-10
2      2026-01-19
3      2026-01-08
4      2025-12-14
          ...    
1493   2017-09-01
1494   2018-10-12
1497   2021-05-05
1498   2019-09-20
1499   2025-03-24
Name: last_air_date, Length: 1485, dtype: datetime64[ns]


In [27]:
# Extract year from premiere date
df_new['premiere_year'] = df_new['first_air_date'].dt.year

In [28]:
print(df_new['premiere_year'])

0       2016
1       2016
2       1993
3       2026
4       1998
        ... 
1493    2017
1494    2017
1497    2021
1498    2019
1499    2024
Name: premiere_year, Length: 1485, dtype: int32


In [29]:
print(df_new.dtypes)

id                                  int64
name                               object
first_air_date             datetime64[ns]
popularity                        float64
vote_average                      float64
vote_count                          int64
status                             object
in_production                        bool
num_seasons                         int64
num_episodes                        int64
genres                             object
type                               object
original_language                  object
origin_country                     object
show_age_days                     float64
days_since_last_episode           float64
keywords                           object
last_air_date              datetime64[ns]
us_content_rating                  object
imdb_id                            object
created_by                         object
homepage                           object
is_canceled                         int64
weighted_rating                   

In [31]:
df_new.to_csv('data/netflix_shows_clean.csv', index=False)
print("Clean dataset saved!")

Clean dataset saved!
